In [ ]:
# --- Papermill parameters (safe defaults) ---
# Tag this cell with: parameters

BASE_FOLDER = "data"
MASTER_CSV = "data/Betfair_Master.csv"
GOOGLE_SHEET_NAME = "Betfair Dashboard"
GOOGLE_SERVICE_ACCOUNT_JSON = ".secrets/google_service_account.json"
PUBLISH = True  # False skips Step 8 (no Google Sheets calls)


In [ ]:
# 📊 Betfair Dashboard
# Built by Gazuty (c) 2025
# This notebook processes Betfair profit/loss data, builds analytics tables, and publishes outputs to Google Sheets.

# --- STEP 0: Configuration ---

import os
from datetime import datetime
from pathlib import Path

# ─── Your Drive / folder paths ───
BASE_FOLDER = os.getenv("BASE_FOLDER", BASE_FOLDER)
MASTER_CSV  = os.getenv("MASTER_CSV", MASTER_CSV)


ARCHIVE_FOLDER  = str(Path(BASE_FOLDER) / "Archive")
BETTING_PATTERN = str(Path(BASE_FOLDER) / "BettingPandL*.csv")

# ─── Google Sheet settings ───
GOOGLE_SHEET_NAME = os.getenv("GOOGLE_SHEET_NAME", GOOGLE_SHEET_NAME)
# ─── Business rules ───
VALID_SPORTS    = ['Horse Racing', 'Greyhound Racing']
MIN_STRIKE_BETS = 50

# ─── Setup ───
os.makedirs(ARCHIVE_FOLDER, exist_ok=True)

print("✅ Configuration loaded:")
print(f"  BASE_FOLDER        = {BASE_FOLDER}")
print(f"  MASTER_CSV         = {MASTER_CSV}")
print(f"  ARCHIVE_FOLDER     = {ARCHIVE_FOLDER}")
print(f"  BETTING_PATTERN    = {BETTING_PATTERN}")
print(f"  GOOGLE_SHEET_NAME  = {GOOGLE_SHEET_NAME}")
print(f"  VALID_SPORTS       = {VALID_SPORTS}")
print(f"  MIN_STRIKE_BETS    = {MIN_STRIKE_BETS}")


In [ ]:
# --- STEP 1: Master Updater ---

import pandas as pd, glob, shutil
from dashboard_lib import combine_exports, parse_money, pick_profit_column, rows_missing_from_master, rows_to_add

REQUIRED_COLS = ['Market', 'Settled date']

def update_betfair_master():
    """Merge new raw exports into the master CSV, verify it on disk, archive them, and return their rows.

    If the saved master is missing any exported bet, this raises before archiving,
    so the exports stay in place and every rerun stops again until it is fixed.
    """
    no_rows = pd.DataFrame(columns=['Market', 'Settled date', 'Profit_Loss'])
    print("\U0001F504 Starting master update")

    # 1⃣ Load or initialize master
    if os.path.exists(MASTER_CSV):
        df_master = pd.read_csv(MASTER_CSV)
        df_master['Settled date'] = pd.to_datetime(df_master['Settled date'], errors='coerce')
        df_master['Profit_Loss'] = pd.to_numeric(df_master['Profit_Loss'], errors='coerce')
        df_master = df_master.dropna(subset=['Settled date']).reset_index(drop=True)
        print(f"\u2705 Loaded master ({len(df_master)} rows)")
    else:
        print("\u26a0 No existing master found — starting fresh")
        # Typed, so the date-based dedup key below works on an empty master
        df_master = pd.DataFrame({
            'Market': pd.Series(dtype=str),
            'Settled date': pd.Series(dtype='datetime64[ns]'),
            'Profit_Loss': pd.Series(dtype=float),
        })

    # 2⃣ Gather raw files
    raw_files = glob.glob(BETTING_PATTERN)
    print(f"📂 Found {len(raw_files)} raw file(s)")

    if not raw_files:
        print("\u26a0 No raw files to process — exiting.")
        return no_rows

    # 3⃣ Process each file
    dfs = []
    for filepath in raw_files:
        fname = os.path.basename(filepath)
        print(f"📅 {fname}", end="")

        df = pd.read_csv(filepath)
        missing = [c for c in REQUIRED_COLS if c not in df.columns]
        if missing:
            print(f" → ❌ missing columns {missing}")
            continue

        pick = pick_profit_column(df.columns)
        if pick is None:
            print(" → ❌ no profit column found")
            continue

        df['Profit_Loss'] = parse_money(df[pick])
        df['Settled date'] = pd.to_datetime(df['Settled date'], errors='coerce')
        df = df[['Market', 'Settled date', 'Profit_Loss']].dropna(subset=['Settled date'])

        dfs.append(df)
        print(f" → {len(df)} rows from '{pick}'")
        unreadable = int(df['Profit_Loss'].isna().sum())
        if unreadable:
            print(
                f"   ⚠ {unreadable} row(s) in {fname} have a P/L that could not be read. They are kept with a blank P/L, "
                "which totals ignore; a corrected re-export adds them again, so fix the master row by hand."
            )

    if not dfs:
        print("\u26a0 No valid data loaded from raw files — exiting.")
        return no_rows

    # 4⃣ Combine exports (a bet in two overlapping downloads counts once) and find what the master lacks
    df_new = combine_exports(dfs)
    print(f"\u2705 {len(df_new)} bet(s) across {len(dfs)} file(s)")
    df_unique = rows_to_add(df_master, df_new)
    print(f"\u2705 {len(df_unique)} new row(s) for the master")

    # 5⃣ Merge, write to a temporary file, and verify it before it replaces the master.
    # A failed or cut-off write then never touches the real master, and the
    # exports are archived only once the new master is known to hold them.
    if not df_unique.empty:
        df_combined = pd.concat([df_master, df_unique], ignore_index=True)
        tmp_csv = f"{MASTER_CSV}.tmp"
        df_combined.to_csv(tmp_csv, index=False)
        saved = pd.read_csv(tmp_csv)
        missing = rows_missing_from_master(saved, df_new)
        if len(saved) != len(df_combined) or not missing.empty:
            if not missing.empty:
                print(missing.to_string(index=False))
            raise RuntimeError(
                f"STOP ❌ — the new master did not save correctly ({len(saved)} of {len(df_combined)} rows read back, "
                f"{int(missing['missing'].sum())} exported row(s) missing). Master and exports left unchanged; not publishing."
            )
        os.replace(tmp_csv, MASTER_CSV)
        print(f"\u2705 Master updated ({len(df_combined)} rows) → {MASTER_CSV}")
    else:
        print("\u26a0 No new rows to add — master unchanged.")
    print(f"\u2705 Verified: all {len(df_new)} exported bet(s) are in the master")

    # 6⃣ Archive files
    for filepath in raw_files:
        fname = os.path.basename(filepath)
        shutil.move(filepath, os.path.join(ARCHIVE_FOLDER, fname))
        print(f"📦 Archived {fname}")

    return df_new

# Run the function; RAW_ROWS feeds the summary check before Step 8
RAW_ROWS = update_betfair_master()


In [ ]:
# --- STEP 2: Load Master ---

import pandas as pd

print(f"📂 Loading master from: {MASTER_CSV}")
df = pd.read_csv(MASTER_CSV)

# Ensure correct dtypes
df['Settled date'] = pd.to_datetime(df['Settled date'], errors='coerce')
df['Profit_Loss'] = pd.to_numeric(df['Profit_Loss'], errors='coerce')

# Drop rows with invalid dates
before = len(df)
df = df.dropna(subset=['Settled date']).reset_index(drop=True)
after = len(df)

print(f"✅ {after} rows loaded (dropped {before - after} invalid dates).")
print(f"   Profit_Loss dtype: {df['Profit_Loss'].dtype}")


In [ ]:
# --- STEP 3: Feature Extraction ---

# 1️⃣ Extract Sport from Market (first token before slash)
df['Sport'] = df['Market'].str.extract(r'^([^/]+)/')[0].str.strip()

# 2️⃣ Extract Track_Info and Event_Description for racing sports
racing_mask = df['Sport'].isin(VALID_SPORTS)
track_event = df.loc[racing_mask, 'Market'].str.extract(r'/\s*(.*?)\s*:\s*(.*)')
track_event.columns = ['Track_Info', 'Event_Description']
df.loc[racing_mask, ['Track_Info', 'Event_Description']] = track_event

# 3️⃣ Extract Country from parentheses in Track_Info
df['Country'] = df['Track_Info'].str.extract(r'\(([^)]+)\)')[0]

# 4️⃣ Fill missing country values
df['Country'] = df['Country'].fillna('UK')
df.loc[~df['Sport'].isin(VALID_SPORTS), 'Country'] = 'Unknown'

# 5️⃣ Clean up Track_Info to produce Track_Name (remove dates and country)
df['Track_Name'] = (
    df['Track_Info']
      .str.replace(r'\([^)]*\)', '', regex=True)
      .str.replace(r'\b\d{1,2}(?:st|nd|rd|th)?\s+\w+\b', '', regex=True)
      .str.strip()
)

# 6️⃣ Preview output
preview = df.loc[df['Track_Name'].notna(), ['Sport', 'Track_Name', 'Country']].drop_duplicates().head(10)
print("✅ Feature extraction complete — first few tracks:")
print(preview)


In [ ]:
# --- STEP 4: Build summary tables (daily, weekly, monthly, sport, country) ---

print("🔧 STEP 4: Building summary tables...")

# 4.1️⃣ Daily Summary (chronological)
by_day = (
    df.groupby(df['Settled date'].dt.date)['Profit_Loss']
      .sum()
      .reset_index(name='Profit_Loss')
      .rename(columns={'Settled date': 'Day'})
)
by_day = by_day.sort_values('Day').reset_index(drop=True)
by_day['Cumulative_Profit_Loss'] = by_day['Profit_Loss'].cumsum()
by_day[['Profit_Loss', 'Cumulative_Profit_Loss']] = by_day[['Profit_Loss', 'Cumulative_Profit_Loss']].round(2)

# 4.2️⃣ Rolling Returns (start from 1 March)
by_day['Day'] = pd.to_datetime(by_day['Day'])
rolling_start = pd.to_datetime('2025-03-01')
rolling_df = by_day[by_day['Day'] >= rolling_start].copy()
rolling_df = rolling_df.set_index('Day')

rolling_df['Rolling 2w'] = by_day.set_index('Day')['Profit_Loss'].rolling(window='14D').sum()
rolling_df['Rolling 4w'] = by_day.set_index('Day')['Profit_Loss'].rolling(window='28D').sum()
rolling_df['Rolling 8w'] = by_day.set_index('Day')['Profit_Loss'].rolling(window='56D').sum()

rolling_df = rolling_df.drop(columns=['Profit_Loss', 'Cumulative_Profit_Loss']).reset_index()
rolling_df = rolling_df.round(2)
rolling_df.columns = ['Day', 'Rolling 2w', 'Rolling 4w', 'Rolling 8w']

# 4.3️⃣ Weekly Summary (weeks run Monday–Sunday, labelled by the Sunday they end on)
by_week = (
    df.set_index('Settled date')
      .resample('W-SUN')['Profit_Loss']
      .sum()
      .reset_index()
      .rename(columns={'Settled date': 'Week Ending'})
)
by_week['Profit_Loss'] = by_week['Profit_Loss'].round(2)

# 4.4️⃣ Rolling by Sport (Horse Racing & Greyhounds)
rolling_by_sport = {}
for sport in ['Horse Racing', 'Greyhound Racing']:
    sport_df = df[df['Sport'] == sport].copy()
    sport_by_day = (
        sport_df.groupby(sport_df['Settled date'].dt.date)['Profit_Loss']
        .sum()
        .reset_index(name='Profit_Loss')
        .rename(columns={'Settled date': 'Day'})
        .sort_values('Day')
        .reset_index(drop=True)
    )
    sport_by_day['Day'] = pd.to_datetime(sport_by_day['Day'])
    sport_by_day = sport_by_day.set_index('Day')

    result = sport_by_day.copy()
    result['Rolling 2w'] = sport_by_day['Profit_Loss'].rolling(window='14D').sum()
    result['Rolling 4w'] = sport_by_day['Profit_Loss'].rolling(window='28D').sum()
    result['Rolling 8w'] = sport_by_day['Profit_Loss'].rolling(window='56D').sum()

    result = result.drop(columns=['Profit_Loss']).reset_index()
    result = result[result['Day'] >= rolling_start].round(2)
    result.columns = ['Day', 'Rolling 2w', 'Rolling 4w', 'Rolling 8w']
    rolling_by_sport[sport] = result

# 4.5️⃣ Monthly Summary — using 'ME' to avoid deprecation warning
by_month = (
    df.set_index('Settled date')
      .resample('ME')['Profit_Loss']
      .sum()
      .reset_index()
)
by_month['Month'] = by_month['Settled date'].dt.to_period('M').astype(str)
by_month = by_month[['Month', 'Profit_Loss']]
by_month['Profit_Loss'] = by_month['Profit_Loss'].round(2)

# 4.6️⃣ Sport Summary
by_sport = (
    df.groupby('Sport')['Profit_Loss']
      .sum()
      .reset_index()
      .round({'Profit_Loss': 2})
)

# 4.7️⃣ Country Summary
by_country = (
    df.groupby('Country')['Profit_Loss']
      .sum()
      .reset_index()
      .round({'Profit_Loss': 2})
)

# 4.8️⃣ Daily Summaries per Sport (with cumulative P/L)
sport_daily = {}
for sport in df['Sport'].dropna().unique():
    temp = (
        df[df['Sport'] == sport]
          .groupby(df['Settled date'].dt.date)['Profit_Loss']
          .sum()
          .reset_index(name='Profit_Loss')
          .rename(columns={'Settled date': 'Day'})
          .sort_values('Day')
          .reset_index(drop=True)
    )
    temp['Cumulative_Profit_Loss'] = temp['Profit_Loss'].cumsum().round(2)
    temp['Profit_Loss'] = temp['Profit_Loss'].round(2)
    sport_daily[f"{sport} Daily"] = temp

# ✅ Summary Checks
print(f"✅ By Day: {len(by_day)} rows (last: {by_day['Day'].max().date()})")
print(f"✅ Rolling Returns: {len(rolling_df)} rows (last: {rolling_df['Day'].max().date()})")
for k, v in rolling_by_sport.items():
    print(f"✅ Rolling {k}: {len(v)} rows (last: {v['Day'].max().date()})")
print(f"✅ By Week: {len(by_week)} rows (last: {by_week['Week Ending'].max().date()})")
print(f"✅ By Month: {len(by_month)} rows (last: {by_month['Month'].max()})")
print(f"✅ By Sport: {len(by_sport)} sports → {by_sport['Sport'].tolist()}")
print(f"✅ By Country: {len(by_country)} countries → {by_country['Country'].tolist()}")


In [ ]:
# --- STEP 5: Track Summaries ---

# 1️⃣ Aggregate P/L per track for Horse and Greyhound Racing
track_df = (
    df[df['Sport'].isin(VALID_SPORTS)]
      .groupby(['Sport', 'Track_Name'], as_index=False)['Profit_Loss']
      .sum()
)
track_df['Profit_Loss'] = track_df['Profit_Loss'].round(2)

# 2️⃣ Create summary groups
tracks = {
    'Track Stats':               track_df,
    'Top Horse Tracks':          track_df.query("Sport == 'Horse Racing'").nlargest(15, 'Profit_Loss'),
    'Bottom Horse Tracks':       track_df.query("Sport == 'Horse Racing'").nsmallest(15, 'Profit_Loss'),
    'Top Greyhound Tracks':      track_df.query("Sport == 'Greyhound Racing'").nlargest(15, 'Profit_Loss'),
    'Bottom Greyhound Tracks':   track_df.query("Sport == 'Greyhound Racing'").nsmallest(15, 'Profit_Loss'),
}

# 3️⃣ Preview sample
print("✅ Track summaries built.")
print(" • Sample Track Stats:")
print(track_df.head())
print(" • Top Horse Tracks:")
print(tracks['Top Horse Tracks'][['Track_Name', 'Profit_Loss']].head())


In [ ]:
# --- STEP 6: Strike Rates ---

# 1️⃣ Filter to Horse & Greyhound Racing
df_racing = df[df['Sport'].isin(VALID_SPORTS)].copy()

# 2️⃣ Compute total bets and wins per track
strike_df = (
    df_racing
      .groupby(['Sport', 'Track_Name'])['Profit_Loss']
      .agg(
          total_bets='count',
          wins=lambda x: (x > 0).sum()
      )
      .reset_index()
)

# 3️⃣ Calculate strike rate
strike_df['Strike_Rate'] = (strike_df['wins'] / strike_df['total_bets']).round(4)

# 4️⃣ Filter by minimum bets threshold
strike_df_filtered = strike_df[strike_df['total_bets'] >= MIN_STRIKE_BETS].reset_index(drop=True)

# 5️⃣ Extract Top & Bottom Strike Rate Tracks
top_strike    = strike_df_filtered.nlargest(10, 'Strike_Rate').reset_index(drop=True)
bottom_strike = strike_df_filtered.nsmallest(10, 'Strike_Rate').reset_index(drop=True)

# 6️⃣ Preview
print(f"✅ Strike rates computed (min {MIN_STRIKE_BETS} bets):")
print("Top 10 Strike Rates:")
print(top_strike[['Sport', 'Track_Name', 'total_bets', 'wins', 'Strike_Rate']])
print("\nBottom 10 Strike Rates:")
print(bottom_strike[['Sport', 'Track_Name', 'total_bets', 'wins', 'Strike_Rate']])


In [ ]:
# --- STEP 7: Prepare all_sheets for export ---

# 7.1️⃣ Ensure all_sheets exists
if 'all_sheets' not in locals():
    all_sheets = {}

# 7.2️⃣ Add core summaries
all_sheets.update({
    'By Day':           by_day,
    'By Day Sorted':    by_day.sort_values('Profit_Loss', ascending=False).reset_index(drop=True),
    'By Week':          by_week,
    'Cumulative':       by_day[['Day', 'Cumulative_Profit_Loss']].rename(columns={'Cumulative_Profit_Loss': 'Cumulative'}),
    'By Month':         by_month,
    'By Sport':         by_sport,
    'By Country':       by_country,
    'Rolling Returns':  rolling_df
})

# 7.3️⃣ Add track-level summaries
all_sheets.update({
    'Track Stats':             tracks['Track Stats'],
    'Top Horse Tracks':        tracks['Top Horse Tracks'],
    'Bottom Horse Tracks':     tracks['Bottom Horse Tracks'],
    'Top Greyhound Tracks':    tracks['Top Greyhound Tracks'],
    'Bottom Greyhound Tracks': tracks['Bottom Greyhound Tracks'],
})

# 7.4️⃣ Add strike rate summaries
all_sheets.update({
    'Top Strike Rates':    top_strike,
    'Bottom Strike Rates': bottom_strike,
})

# 7.5️⃣ Add daily summaries for each sport
all_sheets.update(sport_daily)

# 7.6️⃣ Add rolling returns by sport
for sport, df_rolling in rolling_by_sport.items():
    sheet_name = f"Rolling {sport}"
    all_sheets[sheet_name] = df_rolling

# 7.7️⃣ Final review of included sheets
print(f"✅ Prepared {len(all_sheets)} tables for export:")
for name in all_sheets:
    print(f"  • {name}")


In [ ]:
print("📊 Top Horse Tracks preview:")
print(tracks['Top Horse Tracks'].head())
print("📊 Bottom Horse Tracks preview:")
print(tracks['Bottom Horse Tracks'].head())


In [ ]:
# Quick check before publishing: yesterday/today totals, and every merged row present in the loaded master
import pytz
from datetime import datetime, timedelta
from dashboard_lib import daily_totals, rows_missing_from_master

assert 'df' in globals(), "Please run Steps 2–7 first (df should be in memory)."
assert 'RAW_ROWS' in globals(), "RAW_ROWS not found; please run STEP 1 first."

au = pytz.timezone("Australia/Sydney")
t  = datetime.now(au).date()
y  = t - timedelta(days=1)

master_totals = daily_totals(df, [y, t])
raw_totals    = daily_totals(RAW_ROWS, [y, t])
print(f"MASTER → {y}: {master_totals[y]:.2f} | {t}: {master_totals[t]:.2f}")
print(f" RAWS  → {y}: {raw_totals[y]:.2f} | {t}: {raw_totals[t]:.2f}  (master can be higher: it also holds earlier exports)")

missing = rows_missing_from_master(df, RAW_ROWS)
if not missing.empty:
    print(missing.to_string(index=False))
    raise RuntimeError(f"STOP ❌ — {int(missing['missing'].sum())} merged row(s) are not in the loaded master. Not publishing.")
print(f"PASS ✅ — all {len(RAW_ROWS)} merged row(s) are in the master; proceed to Step 8")


In [ ]:
# --- STEP 8: Export to Google Sheets (local-safe) ---

import os
from pathlib import Path
from datetime import date
import pandas as pd
from dashboard_lib import ensure_grid, retry_on_quota, to_sheet_values

# --- Config ---
SERVICE_JSON = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON", GOOGLE_SERVICE_ACCOUNT_JSON if "GOOGLE_SERVICE_ACCOUNT_JSON" in globals() else str(Path(".secrets") / "google_service_account.json"))
GOOGLE_SHEET_NAME = os.getenv("GOOGLE_SHEET_NAME", GOOGLE_SHEET_NAME if "GOOGLE_SHEET_NAME" in globals() else "Betfair Dashboard")

# scripts/dashboard_charts.gs anchors charts at column D, rows 2–272, each ~18 rows tall.
DASHBOARD_MIN_ROWS, DASHBOARD_MIN_COLS = 300, 12


def write_table(sh, title, frame, APIError, WorksheetNotFound, clear=True, min_rows=0, min_cols=0):
    """Write ``frame`` at A1 of one worksheet, creating or growing it to fit.

    ``clear=False`` leaves the rest of the sheet alone (the Dashboard tab also holds
    the Apps Script charts and anything added by hand). ``min_rows``/``min_cols``
    keep room for things placed beyond the data, such as those charts.
    """
    vals = to_sheet_values(frame)
    n_rows = max(len(vals), min_rows)
    n_cols = max(max(len(r) for r in vals), min_cols)
    try:
        ws = sh.worksheet(title)
    except WorksheetNotFound:
        ws = retry_on_quota(sh.add_worksheet, title=title, rows=n_rows, cols=n_cols, error_type=APIError)
    retry_on_quota(ensure_grid, ws, n_rows, n_cols, error_type=APIError)
    if clear:
        retry_on_quota(ws.clear, error_type=APIError)
    # USER_ENTERED so numbers and ISO dates land as numbers and dates; text cells are apostrophe-prefixed.
    retry_on_quota(ws.update, vals, range_name="A1", value_input_option="USER_ENTERED", error_type=APIError)


def publish():
    import gspread
    from gspread.exceptions import APIError, WorksheetNotFound

    assert "all_sheets" in globals(), "all_sheets not found. Please run Step 7 first."

    gc = gspread.service_account(filename=SERVICE_JSON)
    sh = retry_on_quota(gc.open, GOOGLE_SHEET_NAME, error_type=APIError)
    print(f"✅ Connected to Google Sheet: {GOOGLE_SHEET_NAME}")

    for sheet_name, df_out in all_sheets.items():
        write_table(sh, sheet_name, df_out, APIError, WorksheetNotFound)
        print(f"✅ Updated: {sheet_name} ({len(df_out)} rows)")

    # --- Dashboard KPIs ---
    best = by_day.loc[by_day["Profit_Loss"].idxmax()]
    worst = by_day.loc[by_day["Profit_Loss"].idxmin()]
    kpis = pd.DataFrame(
        [
            ["Total Profit/Loss", round(float(df["Profit_Loss"].sum()), 2)],
            ["Number of Bets", int(len(df))],
            ["Best Day", f'{best["Day"].date()} ({best["Profit_Loss"]:.2f})'],
            ["Worst Day", f'{worst["Day"].date()} ({worst["Profit_Loss"]:.2f})'],
            ["Generated on", date.today()],
        ],
        columns=["Metric", "Value"],
    )
    write_table(
        sh, "Dashboard", kpis, APIError, WorksheetNotFound,
        clear=False, min_rows=DASHBOARD_MIN_ROWS, min_cols=DASHBOARD_MIN_COLS,
    )
    print("✅ Dashboard KPIs updated")


if PUBLISH:
    publish()
else:
    print("⏭ PUBLISH is False — skipped Google Sheets export.")
